# <p align = center> XChem DB Protein Analysis </p>

In [1]:
# Initialize notebook environment
import sys
from pathlib import Path
sys.path.append( Path("../..").resolve().absolute().__str__() )

## Data Processing

#### Extract Sequence Information from PDB

In [ ]:
import gemmi
sessionPath = Path("../../../data/s3Data/2017_lb18145-3" )
dimplePDBPath = Path( "../../../data/s3Data/2017_lb18145-3/000-load/dimple.pdb").resolve().__str__() # .absolute()

pdb = gemmi.read_pdb(dimplePDBPath)
pdb.setup_entities()
pdbModel = pdb[0] 

subchains =  [subchain for subchain in pdbModel.subchains() if subchain.check_polymer_type().name == "PeptideL"] 
seqs = [subchain.make_one_letter_sequence() for subchain in subchains]
fastaSeqs = [  "".join( [gemmi.find_tabulated_residue(resname).fasta_code() for resname in subchain.extract_sequence() ] )for subchain in subchains ]

print( seqs)
print( fastaSeqs )

['SMLDDAKARLRKYDIGGKYSHLPYNKYSVLLPLVAKEGKLHLLFTVRSEKLRRAPGEVCFPGGKRDPTDMDDAATALREAQEEVGLRPHQVEVVCCLVPCLIDTDTLITPFVGLIDHNFQAQPNPAEVKDVFLVPLAYFLHPQVHDQ-INHIFEYTNPEDGVTYQIKGMTANLAVLVAFIILEKKPT']
['SMLDDAKARLRKYDIGGKYSHLPYNKYSVLLPLVAKEGKLHLLFTVRSEKLRRAPGEVCFPGGKRDPTDMDDAATALREAQEEVGLRPHQVEVVCCLVPCLIDTDTLITPFVGLIDHNFQAQPNPAEVKDVFLVPLAYFLHPQVHDQINHIFEYTNPEDGVTYQIKGMTANLAVLVAFIILEKKPT']


#### Blast API with BioPython

Documents: [link](https://biopython.org/docs/dev/Tutorial/chapter_blast.html)

Blast Options:
-  `blastp` (prot-prot) | `blastn` (nucleo-nucleo) |  `blastx` (translated nuc > prot) | `tblast` (prot > translated nuc) | `tblastx` ( translated nuc > translated nuc )

Data base Options:
- Proteins: 
    - `nr` (non-redundant prot seq) | `swissprot` | ...
- 
    - `core_nt`, 

Output Formats:
- `XML` (default) |`HTML` | `Text` | `Tabular` | `JSON2` | `XML2` 


In [ ]:
from pathlib import Path
from Bio import Blast
import json
Blast.email ="alex.belo@ed.ac.uk"

sessionPath = Path("../../../data/s3Data/2017_lb18145-3" )
savePath = sessionPath / "protein" / "blast_result.json"
savePath.parent.mkdir(parents=True, exist_ok=True) 

aaseq =  "SMLDDAKARLRKYDIGGKYSHLPYNKYSVLLPLVAKEGKLHLLFTVRSEKLRRAPGEVCFPGGKRDPTDMDDAATALREAQEEVGLRPHQVEVVCCLVPCLIDTDTLITPFVGLIDHNFQAQPNPAEVKDVFLVPLAYFLHPQVHDQINHIFEYTNPEDGVTYQIKGMTANLAVLVAFIILEKKPT"
result_stream = Blast.qblast("blastp", "nr", aaseq, format_type="JSON2")
print(result_stream.getheaders())

data = result_stream.read()

with open( savePath.resolve().__str__() , "wb") as out_stream:
    out_stream.write( result_stream.read() )
result_stream.close()  # Close the stream after reading


In [ ]:
import io
import zipfile

# data -> This is a byte string containing the BLAST result in ZIP format.
# So first, I need to load the data into memory as a byte stream, so that it can be processed as a ZIP file.
# Then, I need to unzip the data. By doing so, I will get access to some metadata, such as the name of the files that have been zipped.
# Then, I pick the file what I want to read from the ZIP archive access its specific 

with open( savePath.resolve().__str__() , "rb") as f:
    data = f.read()  # Read the content of the file as bytes in a stream-like fashion

# The BLAST result is a ZIP file, not a plain JSON or text file.
with zipfile.ZipFile( io.BytesIO(data) ) as zf: # -> Load the ZIP file from the byte stream into memory as a zipfile json object

    print( zf.namelist() )  # -> THere are two files in the ZIP archive, one is the metadata and the other is the actual JSON data
    fileName = zf.namelist()[0]  
    fileData = zf.namelist()[1]  

    with zf.open(fileData) as f: # - > Select the second file in the ZIP archive object
        new_data = f.read() # -> Read the content of the file as bytes in a stream-like fashion
        final_data = new_data.decode("utf-8") # -> Decode the byte string to a regular string
        print(type( final_data ))  
        result = json.loads(final_data) # -> Parse the JSON data into a Python dictionary
        print(result)



# print(result)

['7N7N0K1D014.json', '7N7N0K1D014_1.json']
<class 'str'>
{'BlastOutput2': {'report': {'program': 'blastp', 'version': 'BLASTP 2.17.0+', 'reference': 'Stephen F. Altschul, Thomas L. Madden, Alejandro A. Sch&auml;ffer, Jinghui Zhang, Zheng Zhang, Webb Miller, and David J. Lipman (1997), "Gapped BLAST and PSI-BLAST: a new generation of protein database search programs", Nucleic Acids Res. 25:3389-3402.', 'search_target': {'db': 'nr'}, 'params': {'matrix': 'BLOSUM62', 'expect': 10, 'gap_open': 11, 'gap_extend': 1, 'filter': 'F', 'cbs': 2}, 'results': {'search': {'query_id': 'Query_135097', 'query_title': 'unnamed protein product', 'query_len': 186, 'hits': [{'num': 1, 'description': [{'id': 'pdb|5T3P|A', 'accession': '5T3P_A', 'title': 'Chain A, Peroxisomal coenzyme A diphosphatase NUDT7 [Homo sapiens]', 'taxid': 9606, 'sciname': 'Homo sapiens'}, {'id': 'pdb|5T3P|B', 'accession': '5T3P_B', 'title': 'Chain B, Peroxisomal coenzyme A diphosphatase NUDT7 [Homo sapiens]', 'taxid': 9606, 'scinam

In [ ]:
id = 0

In [ ]:
print( result.keys())
print( result["BlastOutput2"].keys() )
print( result["BlastOutput2"]["report"].keys() )# -> Get the title of the first hit in the BLAST result
print( result["BlastOutput2"]["report"]["results"].keys() )
print( result["BlastOutput2"]["report"]["results"]["search"].keys() )
print( result["BlastOutput2"]["report"]["results"]["search"]["hits"] )
print( result["BlastOutput2"]["report"]["results"]["search"]["hits"][0].keys() )
print( result["BlastOutput2"]["report"]["results"]["search"]["hits"][0]["description"][0].keys() ) #  HSPs = (High-scoring Segment Pairs)
print( result["BlastOutput2"]["report"]["results"]["search"]["hits"][id]["description"][0]["id"] )
print( result["BlastOutput2"]["report"]["results"]["search"]["hits"][id]["description"][0]["sciname"] )
id +=1

dict_keys(['BlastOutput2'])
dict_keys(['report'])
dict_keys(['program', 'version', 'reference', 'search_target', 'params', 'results'])
dict_keys(['search'])
dict_keys(['query_id', 'query_title', 'query_len', 'hits', 'stat'])
[{'num': 1, 'description': [{'id': 'pdb|5T3P|A', 'accession': '5T3P_A', 'title': 'Chain A, Peroxisomal coenzyme A diphosphatase NUDT7 [Homo sapiens]', 'taxid': 9606, 'sciname': 'Homo sapiens'}, {'id': 'pdb|5T3P|B', 'accession': '5T3P_B', 'title': 'Chain B, Peroxisomal coenzyme A diphosphatase NUDT7 [Homo sapiens]', 'taxid': 9606, 'sciname': 'Homo sapiens'}, {'id': 'pdb|5T3P|C', 'accession': '5T3P_C', 'title': 'Chain C, Peroxisomal coenzyme A diphosphatase NUDT7 [Homo sapiens]', 'taxid': 9606, 'sciname': 'Homo sapiens'}], 'len': 236, 'hsps': [{'num': 1, 'bit_score': 376.326, 'score': 965, 'evalue': 2.41004e-130, 'identity': 185, 'positive': 186, 'query_from': 1, 'query_to': 186, 'hit_from': 15, 'hit_to': 210, 'align_len': 196, 'gaps': 10, 'qseq': 'SMLDDAKARLRKYDIGGK

In [ ]:
import re
blastID = result["BlastOutput2"]["report"]["results"]["search"]["hits"][0]["description"][0]["id"]
print(blastID)  # Get the BLAST ID of the first hit in the BLAST result
pdbID = re.search( "^pdb\|\w+\|", blastID ).group()[4:8]  # Extract the PDB ID from the BLAST ID using a regular expression
print(pdbID)  # Print the extracted PDB ID

pdb|5T3P|A
5T3P


In [ ]:
pdbID

'pdb'

In [ ]:
# Debug: Check what type of data we have
print("Data type:", type(data))
print("Data length:", len(data))
print("First 20 bytes:", data[:20])
print("Is it a ZIP file?", data[:2] == b'PK')  # ZIP files start with 'PK'

Data type: <class 'bytes'>
Data length: 0
First 20 bytes: b''
Is it a ZIP file? False


## Get fasta sequence for the 4 sessions that I have access to

In [ ]:
# Make way to access the 20 pbds of interest
import re
import shutil

rootDirPath = Path( "../../../data/s3Data")
for sessionDir in rootDirPath.iterdir():
    if sessionDir.is_dir() and re.search( "^[0-9][0-9][0-9][0-9]", sessionDir.name): # Access only the Session Dir folders
        for datasetDir in sessionDir.iterdir(): # Access only the crystal dataset
            if datasetDir.is_dir() and re.search( "-x[0-9]+$", datasetDir.name):
                dimplePDBDir = datasetDir / "dimple.pdb"
                for pdbFile in dimplePDBDir.iterdir()
                    if pdbFile.is_file:

                        newPath = rootDirPath / sessionDir.name / 
                        shutil.copy(pdbFile, )






In [6]:
from pathlib import Path
import shutil

shutil.copy(Path("./test2.txt"), Path("../s3Analysis/test.txt"))


WindowsPath('../s3Analysis/test.txt')

## Get the Blast Results (top match with pdb code)

## Look at Results of 20 PDB codes

## Protein Diversity

### Protein Sequence Diversity

### Protein Structure Diversity

### Protein Physical Properties Diversity

I.e. ( pI, )